# Vertex AI — Gen AI Hands-On Lab
### GCP Training Program — Day 4, Module 6: Vertex Platform — Gen AI

**What this notebook covers:** Gemini models (text, chat, multimodal, function calling), a small
Vector Search index for grounding/RAG, Google Search grounding, and a guided walkthrough of Model
Garden and Agent Builder.

## ⚠️ Cost & trial-account notes
Good news for this notebook specifically: **Gemini API calls are fast and cheap** (fractions of a
cent per call) — this is the most "live-hands-on-friendly" section of the whole Day 4-5 curriculum.
The one resource that bills continuously here is the **Vector Search index endpoint** in Section 6 —
it bills per node-hour once deployed, so **undeploy/delete it right after the demo**, not at the
end of the day.

**This notebook is fully self-contained.** Authenticate → Configure → everything else runs
against your own project.


## 1. Authenticate This Session

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures gcloud for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth application-default login")

## 2. Install & Import
The `vertexai` module (part of `google-cloud-aiplatform`) covers Gemini, embeddings, and grounding.

In [ ]:
!pip install --quiet "google-cloud-aiplatform>=1.60.0"

In [ ]:
import time
import os
import vertexai
from vertexai.generative_models import GenerativeModel, Part, Tool, grounding, GenerationConfig
from vertexai.language_models import TextEmbeddingModel

print("vertexai imported successfully")

## 3. Configure Your Project & Region

In [ ]:
PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your Vertex AI region [default: us-central1]: ").strip()
if not REGION:
    REGION = "us-central1"

_suffix = int(time.time())
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION

!gcloud config set project {PROJECT_ID}

vertexai.init(project=PROJECT_ID, location=REGION)

# Centralized model names — defined once, used everywhere below. Gemini/embedding model IDs are
# retired and replaced faster than most other GCP resources; if either of these throws a
# NotFound/InvalidArgument error when first used, check current available model IDs with:
#   gcloud ai models list --region={REGION} --filter="displayName:gemini"
# and update the two lines below rather than hunting for hardcoded strings elsewhere in this notebook.
GEMINI_MODEL_NAME = "gemini-1.5-flash"
EMBEDDING_MODEL_NAME = "text-embedding-004"

print("Vertex AI Gen AI SDK initialized.")
print("Gemini model:", GEMINI_MODEL_NAME)
print("Embedding model:", EMBEDDING_MODEL_NAME)

## 4. Gemini Models on Vertex AI
### 4.1 Single-Turn Text Generation
**What this does:** the simplest possible Gemini call — one prompt in, one response out, no
conversation history maintained.

In [ ]:
model = GenerativeModel(GEMINI_MODEL_NAME)

response = model.generate_content("Explain what a data lake is, in two sentences, for a business audience.")
print(response.text)

### 4.2 Multi-Turn Chat
**What this does:** `start_chat()` keeps conversation history automatically — each `send_message()`
call includes everything said before it, so the second question below can reference "it" and the
model resolves that to the topic from the first turn.

In [ ]:
chat = model.start_chat()

r1 = chat.send_message("What is BigQuery partitioning?")
print("Turn 1:", r1.text[:300])

r2 = chat.send_message("Now explain how it differs from clustering, referencing what you just said.")
print("\nTurn 2:", r2.text[:400])

### 4.3 Multimodal Input (Text + Image)
**What this does:** Gemini natively accepts image input alongside text in the same call —
`Part.from_uri` references an image directly from a public GCS URI without needing to download it
first.

In [ ]:
image_part = Part.from_uri(
    "gs://cloud-samples-data/generative-ai/image/scones.jpg",
    mime_type="image/jpeg",
)

response = model.generate_content([image_part, "What is in this image? Answer in one sentence."])
print(response.text)

### 4.4 Function Calling (Tool Use) — a "Poor Man's Agent"
**What this does:** rather than the model just producing text, we give it a *tool definition* (a
function signature with no actual implementation shown to the model). When the prompt needs that
capability, Gemini returns a structured function-call request instead of prose — your code executes
the real function and can feed the result back in. This is the same underlying mechanism full agent
frameworks (including Agent Builder, Section 7) are built on.

In [ ]:
from vertexai.generative_models import FunctionDeclaration

get_station_trip_count = FunctionDeclaration(
    name="get_station_trip_count",
    description="Get the number of bike trips that started at a given station name.",
    parameters={
        "type": "object",
        "properties": {
            "station_name": {"type": "string", "description": "The bike station name."},
        },
        "required": ["station_name"],
    },
)

trip_tool = Tool(function_declarations=[get_station_trip_count])
tool_model = GenerativeModel(GEMINI_MODEL_NAME, tools=[trip_tool])

response = tool_model.generate_content("How many trips started at the Riverside station?")

function_call = response.candidates[0].content.parts[0].function_call
print("Gemini requested a function call:")
print(" name:", function_call.name)
print(" args:", dict(function_call.args))
print("\n(Your code would now actually look this up and send the result back — that round trip")
print("is exactly what Agent Builder automates for you, see Section 7.)")

## 5. Grounding With Google Search
**What this does:** rather than answering purely from training data, `grounding.GoogleSearchRetrieval`
lets Gemini issue real Google Search queries and ground its answer in current results — the fastest,
cheapest way to demo grounding, with no infrastructure to deploy (contrast with Section 6, where we
ground against *our own* documents instead of the public web).

In [ ]:
# Caveat: Vertex AI's grounding tool API has shifted across SDK versions — older models used
# Tool.from_google_search_retrieval(grounding.GoogleSearchRetrieval()), newer ones use
# Tool.from_google_search(grounding.GoogleSearch()). Try the current shape first, fall back to the
# older one so this cell keeps working regardless of which your installed SDK version expects.
try:
    search_tool = Tool.from_google_search(grounding.GoogleSearch())
except AttributeError:
    search_tool = Tool.from_google_search_retrieval(grounding.GoogleSearchRetrieval())

grounded_model = GenerativeModel(GEMINI_MODEL_NAME, tools=[search_tool])

response = grounded_model.generate_content("What GCP announcements happened in the last week?")
print(response.text)

if response.candidates[0].grounding_metadata.grounding_chunks:
    print("\nGrounding sources used:")
    for chunk in response.candidates[0].grounding_metadata.grounding_chunks:
        if chunk.web:
            print(" -", chunk.web.title, "|", chunk.web.uri)

## 6. Vector Search — Grounding on Your Own Documents (RAG)
### 6.1 Embed a Handful of Small Documents
**What this does:** converts each short text document into a numeric embedding vector — documents
with similar meaning end up with similar vectors, which is what makes semantic search possible.
We use only 5 tiny documents to keep index build time and cost minimal.

In [ ]:
documents = [
    "Cloud Storage stores objects in buckets with configurable storage classes and lifecycle rules.",
    "BigQuery partitioning splits a table by date or integer range to reduce bytes scanned per query.",
    "Pub/Sub delivers messages to every subscription attached to a topic — this is the fan-out pattern.",
    "Vertex AI Feature Store centralizes features for consistent use across training and serving.",
    "Dataflow runs Apache Beam pipelines for both batch and streaming data processing on GCP.",
]

embedding_model = TextEmbeddingModel.from_pretrained(EMBEDDING_MODEL_NAME)
embeddings = embedding_model.get_embeddings(documents)

print(f"Generated {len(embeddings)} embeddings, each with {len(embeddings[0].values)} dimensions.")

### 6.2 Build & Deploy a Small Vector Search Index
**Cost/time note:** this is the one resource in this notebook that bills per node-hour once
deployed. `e2-standard-2` (the smallest supported endpoint machine type) and a tiny 5-document
index keep both build time (a few minutes) and ongoing cost as low as possible. **Undeploy and
delete this right after the demo (Section 6.4) — don't leave it running.**

In [ ]:
import json

os.makedirs("vector_search_data", exist_ok=True)
with open("vector_search_data/embeddings.json", "w") as f:
    for i, emb in enumerate(embeddings):
        f.write(json.dumps({"id": str(i), "embedding": emb.values}) + "\n")

EMBEDDINGS_BUCKET = f"gs://{PROJECT_ID}-vector-search-{_suffix}"
!gsutil mb -l {REGION} {EMBEDDINGS_BUCKET}
!gsutil cp vector_search_data/embeddings.json {EMBEDDINGS_BUCKET}/embeddings.json

In [ ]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION)

index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=f"training-doc-index-{_suffix}",
    contents_delta_uri=EMBEDDINGS_BUCKET,
    dimensions=len(embeddings[0].values),
    approximate_neighbors_count=5,
    distance_measure_type="DOT_PRODUCT_DISTANCE",
    index_update_method="BATCH_UPDATE",
    # Explicit small-corpus-safe values — defaults are tuned for much larger indexes and can
    # behave oddly (e.g. empty results) on a toy 5-document index like this one.
    leaf_node_embedding_count=5,
    leaf_nodes_to_search_percent=100,
)
print("Created index:", index.resource_name)

index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name=f"training-doc-endpoint-{_suffix}",
    public_endpoint_enabled=True,
)

DEPLOYED_INDEX_ID = f"deployed_{_suffix}"
index_endpoint.deploy_index(
    index=index,
    deployed_index_id=DEPLOYED_INDEX_ID,
    machine_type="e2-standard-2",
    min_replica_count=1,
    max_replica_count=1,
)
print("Deployed index endpoint:", index_endpoint.resource_name)

### 6.3 Query the Index & Ground a Gemini Answer
**What this does:** embed the *question* with the same embedding model, find the most similar
stored document via Vector Search, then inject that document's text directly into the Gemini
prompt as context — the core pattern behind Retrieval-Augmented Generation (RAG).

In [ ]:
query_text = "How does BigQuery reduce the amount of data scanned per query?"
query_embedding = embedding_model.get_embeddings([query_text])[0].values

neighbors = index_endpoint.find_neighbors(
    deployed_index_id=DEPLOYED_INDEX_ID,
    queries=[query_embedding],
    num_neighbors=2,
)

retrieved_docs = [documents[int(n.id)] for n in neighbors[0]]
print("Retrieved context:", retrieved_docs)

grounded_prompt = f"""Answer the question using ONLY the context below.

Context:
{chr(10).join(retrieved_docs)}

Question: {query_text}"""

rag_response = model.generate_content(grounded_prompt)
print("\nGrounded answer:\n", rag_response.text)

### 6.4 Undeploy & Delete the Index — Do This Now
This is the billed resource from this section — clean it up as soon as the demo is done.

In [ ]:
index_endpoint.undeploy_index(deployed_index_id=DEPLOYED_INDEX_ID)
index_endpoint.delete(force=True)
index.delete()
!gsutil -m rm -r {EMBEDDINGS_BUCKET}
print("Vector Search index, endpoint, and staging bucket deleted.")

## 7. Model Garden (Guided Walkthrough)
**Why this is a walkthrough, not a live exercise:** Model Garden is primarily a browsing/deployment
UI in the console — deploying any of its open models (Llama, Gemma, etc.) creates a dedicated
endpoint that bills per node-hour, same as Section 6's Vector Search endpoint, but typically on a
larger and more expensive machine type. Walk through the console instead of deploying live:

- Model Garden catalogs first-party (Gemini), open (Gemma, Llama), and third-party partner models
  (including Anthropic's Claude models, available as a partner model in supported regions).
- Deployment options vary by model: some deploy to a dedicated Vertex AI endpoint (node-hour
  billing), others are available as a pay-per-token API call with zero idle cost — worth pointing
  out to the class as the key cost-model distinction to check before deploying anything.
- If you do want a live moment here, `gemma-2b` on the smallest supported machine type is the
  lowest-cost open-model deployment option — but confirm current pricing before committing to it
  live, since machine-type requirements per model change over time.

In [ ]:
!gcloud ai model-garden models list --region={REGION} --limit=10

## 8. Agent Builder (Guided Walkthrough)
**Why this is a walkthrough, not a live exercise:** Agent Builder is primarily console-driven —
defining an agent's goal, tools, and data stores happens through the UI, with programmatic control
available but not the primary intended path. Walk through the concept using what Section 4.4 just
demonstrated:

- **The core idea:** an agent is a loop around exactly the function-calling mechanism from
  Section 4.4 — the model decides which tool to call and with what arguments, your code (or Agent
  Builder's managed runtime) executes it, the result goes back to the model, and this repeats until
  the model produces a final answer instead of another tool call.
- **What Agent Builder adds on top:** a managed orchestration loop (so you don't hand-write the
  "call tool → feed result back → repeat" loop yourself), grounding against your own data stores,
  and a conversational UI/widget you can embed, all with less custom code than wiring the loop up
  by hand.
- **Cost model:** billed per conversation/query, not per node-hour — closer to the Gemini API's
  cost profile than to a deployed endpoint, making it comfortable for a follow-up class exercise
  outside a live session if you want participants to try building one on their own trial account.

## 9. Cleanup & Final Verification
Section 6.4 already deletes the one resource in this notebook that bills continuously (the Vector
Search index endpoint) — this section is a final safety-net check, not a second cleanup pass.
Sections 7 and 8 are guided walkthroughs only (a read-only `gcloud ai model-garden models list` and
no code execution at all) — nothing to delete there **unless you personally deployed a Model Garden
model live**, in which case that endpoint is not tracked by this notebook and needs manual deletion.

In [ ]:
print("Checking for any lingering Vector Search indexes/endpoints from this notebook...")
!gcloud ai indexes list --region={REGION} --filter="displayName:training-doc-index" --format="value(name)"
!gcloud ai index-endpoints list --region={REGION} --filter="displayName:training-doc-endpoint" --format="value(name)"

print("\nBoth commands above should return nothing if Section 6.4 ran successfully.")
print("If either lists something, delete it with:")
print("  gcloud ai index-endpoints delete ENDPOINT_ID --region={REGION} --quiet   (undeploy first if needed)")
print("  gcloud ai indexes delete INDEX_ID --region={REGION} --quiet")

print("\nChecking for any lingering buckets created by this notebook...")
!gsutil ls -p {PROJECT_ID} | grep -i "vector-search" || echo "None found."

print("\nReminder: if you deployed a real Model Garden model in Section 7 (outside this notebook's")
print("tracked variables), check Vertex AI > Online prediction > Endpoints in the console and delete")
print("it manually — that resource isn't created or tracked by any cell here.")